<a href="https://colab.research.google.com/github/robertwzhou/Capstone3_fakenews/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('news.csv')

# Assuming columns are named 'text' and 'label' based on your description.
# If the column names are different, please adjust them here.
# Convert boolean labels (True/False) to integers (1/0)
df['label'] = df['label'].astype(int)

# Split the data into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print("Data is ready for tokenization.")

Training samples: 36605
Validation samples: 9152
Data is ready for tokenization.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

# Using TinyBERT (4 layers, 312 hidden size) - much smaller than DistilBERT
model_checkpoint = "huawei-noah/TinyBERT_General_4L_312D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Prepare Dataset
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, padding=True, truncation=True, max_length=128) # Shorter max_length for speed
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_texts, train_labels)
val_dataset = NewsDataset(val_texts, val_labels)

# Load small model
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# Training Arguments optimized for speed
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,           # Reduced epochs for initial run
    per_device_train_batch_size=32, # Larger batch size for faster processing
    eval_strategy="epoch",
    save_strategy="no",           # Don't waste time saving checkpoints
    fp16=torch.cuda.is_available() # Use mixed precision if GPU is available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("Small model (TinyBERT) loaded. Run trainer.train() next.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Small model (TinyBERT) loaded. Run trainer.train() next.


In [ ]:
# Execute the transfer learning training loop
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.035018,0.027517


TrainOutput(global_step=1144, training_loss=0.06796830642473448, metrics={'train_runtime': 56.2607, 'train_samples_per_second': 650.632, 'train_steps_per_second': 20.334, 'total_flos': 131219739194880.0, 'train_loss': 0.06796830642473448, 'epoch': 1.0})

In [ ]:
# Evaluate the model on the validation set
evaluation_results = trainer.evaluate()
print(evaluation_results)

Training Loss,Validation Loss,Epoch
0.035018,0.027517,1


{'eval_loss': 0.02751653455197811}


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# Get predictions on the validation set
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

# Display metrics
print(f'Accuracy: {accuracy_score(val_labels, preds):.4f}')
print('\nClassification Report:')
print(classification_report(val_labels, preds, target_names=['False (0)', 'True (1)']))

Accuracy: 0.9928

Classification Report:
              precision    recall  f1-score   support

   False (0)       0.99      0.99      0.99      4606
    True (1)       0.99      0.99      0.99      4546

    accuracy                           0.99      9152
   macro avg       0.99      0.99      0.99      9152
weighted avg       0.99      0.99      0.99      9152



In [ ]:
new_model_checkpoint = "distilbert/distilroberta-base"
new_tokenizer = AutoTokenizer.from_pretrained(new_model_checkpoint)

# Re-tokenize with the new tokenizer
class NewNewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = new_tokenizer(texts, padding=True, truncation=True, max_length=128)
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

new_train_dataset = NewNewsDataset(train_texts, train_labels)
new_val_dataset = NewNewsDataset(val_texts, val_labels)

# Initialize DistilRoBERTa
new_model = AutoModelForSequenceClassification.from_pretrained(new_model_checkpoint, num_labels=2)

new_trainer = Trainer(
    model=new_model,
    args=training_args,
    train_dataset=new_train_dataset,
    eval_dataset=new_val_dataset
)

print(f'Ready to train {new_model_checkpoint}. Run new_trainer.train() to begin.')

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Ready to train distilbert/distilroberta-base. Run new_trainer.train() to begin.


In [ ]:
# Train the comparison model (DistilRoBERTa)
new_trainer.train()

# Evaluate and print results
new_results = new_trainer.evaluate()
print(f'DistilRoBERTa Evaluation Results: {new_results}')

Epoch,Training Loss,Validation Loss
1,0.007546,0.006264


Training Loss,Validation Loss,Epoch
0.007546,0.006264,1


DistilRoBERTa Evaluation Results: {'eval_loss': 0.0062638213858008385}


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# Get predictions for DistilRoBERTa
new_predictions = new_trainer.predict(new_val_dataset)
new_preds = np.argmax(new_predictions.predictions, axis=-1)

# Display metrics
print(f'DistilRoBERTa Accuracy: {accuracy_score(val_labels, new_preds):.4f}')
print('\nDistilRoBERTa Classification Report:')
print(classification_report(val_labels, new_preds, target_names=['False (0)', 'True (1)']))

DistilRoBERTa Accuracy: 0.9986

DistilRoBERTa Classification Report:
              precision    recall  f1-score   support

   False (0)       1.00      1.00      1.00      4606
    True (1)       1.00      1.00      1.00      4546

    accuracy                           1.00      9152
   macro avg       1.00      1.00      1.00      9152
weighted avg       1.00      1.00      1.00      9152



Tried training another model, but session crashed after using all available RAM